# Taller 6 — Redes Neuronales Convolucionales (CNN) con CIFAR-10

**Asignatura:** Tópicos en Inteligencia Artificial (TIA)  
**Tema:** Deep Learning — Redes Convolucionales  
**Dataset:** CIFAR-10  
**Framework:** PyTorch  

---

## Objetivos
1. Diseñar y entrenar una CNN configurable sobre CIFAR-10.
2. Aplicar preprocesamiento, normalización y *data augmentation* justificados.
3. Realizar búsqueda aleatoria de hiperparámetros (*Random Search*).
4. Evaluar el modelo final con métricas completas y análisis de sobreajuste.
5. Comparar CNN vs MLP y responder las preguntas de análisis del taller.


In [ ]:
# ── Instalación automática de dependencias ──────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch',
    'torchvision',
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
        print(f'  {pkg} ya instalado.')
    except ImportError:
        print(f'  Instalando {pkg}...')
        install(pkg)

print('\nTodas las dependencias están listas.')


In [ ]:
# ── Importaciones globales y configuración ──────────────────────────────────
import os
import time
import random
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, ConfusionMatrixDisplay
)
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings('ignore')

# ── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dispositivo ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Nombres de clases en español ────────────────────────────────────────────
CLASS_NAMES = [
    'Avión', 'Automóvil', 'Pájaro', 'Gato', 'Ciervo',
    'Perro', 'Rana', 'Caballo', 'Barco', 'Camión'
]

BATCH_SIZE = 128
print(f'Clases: {CLASS_NAMES}')


---
## Parte 1 — Preprocesamiento, Normalización y Data Augmentation

### 1.1 Normalización
CIFAR-10 tiene imágenes RGB de 32×32 píxeles con valores en [0, 255].  
Se normalizan con la media y desviación estándar calculadas sobre el conjunto de entrenamiento completo:

| Canal | Media  | Desv. Est. |
|-------|--------|------------|
| R     | 0.4914 | 0.2470     |
| G     | 0.4822 | 0.2435     |
| B     | 0.4465 | 0.2616     |

La normalización centra la distribución de activaciones, acelera la convergencia y mejora la estabilidad numérica del gradiente.

### 1.2 Data Augmentation — Justificación

| Transformación | Parámetros | Justificación |
|---|---|---|
| `RandomHorizontalFlip` | p=0.5 | Los objetos de CIFAR-10 son simétricos horizontalmente; duplica la variedad sin distorsión semántica. |
| `RandomCrop` | size=32, padding=4 | Simula pequeñas traslaciones; el modelo aprende invarianza posicional. |
| `ColorJitter` | brightness=0.2, contrast=0.2, saturation=0.2 | Robustez ante variaciones de iluminación y condiciones de captura. |
| `RandomRotation` | degrees=15 | Invarianza a pequeñas rotaciones presentes en imágenes reales. |
| `ToTensor` | — | Convierte PIL Image [0,255] a tensor float [0,1]. |
| `Normalize` | mean, std por canal | Estandariza la entrada; mejora convergencia y estabilidad. |

> **Nota:** El conjunto de validación/test **no** recibe augmentation; solo `ToTensor` + `Normalize`.


In [ ]:
# ── Carga de CIFAR-10, split estratificado 80/20 y DataLoaders ──────────────

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

# Transformaciones de entrenamiento (con augmentation)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Transformaciones de validación/test (sin augmentation)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Descargar dataset completo (sin transform para obtener etiquetas)
full_train_raw = datasets.CIFAR10(root='./data', train=True,  download=True, transform=None)
test_dataset   = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# Split estratificado 80/20 sobre el conjunto de entrenamiento
labels = np.array(full_train_raw.targets)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

print(f'Tamaño entrenamiento : {len(train_idx):,}')
print(f'Tamaño validación    : {len(val_idx):,}')
print(f'Tamaño test          : {len(test_dataset):,}')

# Crear datasets con las transformaciones correctas
train_dataset_aug = datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
train_dataset_raw = datasets.CIFAR10(root='./data', train=True, download=False, transform=val_transform)

train_subset = Subset(train_dataset_aug, train_idx)
val_subset   = Subset(train_dataset_raw, val_idx)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f'\nBatches por época (train): {len(train_loader)}')
print(f'Batches por época (val)  : {len(val_loader)}')


In [ ]:
# ── EDA: distribución de clases, mosaico de imágenes y ejemplos de augmentation ──

# 1. Distribución de clases en train y val
train_labels = labels[train_idx]
val_labels   = labels[val_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, lbl, title in zip(axes, [train_labels, val_labels], ['Entrenamiento', 'Validación']):
    counts = np.bincount(lbl)
    bars = ax.bar(CLASS_NAMES, counts, color=sns.color_palette('tab10', 10))
    ax.set_title(f'Distribución de clases — {title}', fontsize=13)
    ax.set_xlabel('Clase')
    ax.set_ylabel('Cantidad de imágenes')
    ax.tick_params(axis='x', rotation=45)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                str(cnt), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

# 2. Mosaico de imágenes originales (una por clase)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()
shown = set()
idx_ptr = 0
raw_data = full_train_raw
for i in range(len(raw_data)):
    img, lbl = raw_data[i]
    if lbl not in shown:
        axes[lbl].imshow(img)
        axes[lbl].set_title(CLASS_NAMES[lbl], fontsize=11)
        axes[lbl].axis('off')
        shown.add(lbl)
    if len(shown) == 10:
        break
fig.suptitle('Ejemplo de imagen por clase (original)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# 3. Ejemplos de data augmentation (misma imagen, 8 versiones)
aug_transform_vis = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(degrees=15),
])

sample_img, sample_lbl = full_train_raw[0]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
axes[0].imshow(sample_img)
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')
for k in range(1, 8):
    aug_img = aug_transform_vis(sample_img)
    axes[k].imshow(aug_img)
    axes[k].set_title(f'Aug #{k}', fontsize=10)
    axes[k].axis('off')
fig.suptitle(f'Data Augmentation — clase: {CLASS_NAMES[sample_lbl]}', fontsize=13)
plt.tight_layout()
plt.show()


---
## Parte 2 — Diseño de la Arquitectura CNN

### 2.1 Justificación arquitectónica

| Componente | Decisión | Justificación |
|---|---|---|
| **Bloques Conv** | 2–3 bloques apilados | Más bloques = mayor campo receptivo; 2–3 es suficiente para imágenes 32×32. |
| **Doble convolución por bloque** | Conv-BN-ReLU × 2 | Aumenta la capacidad representacional sin aumentar el campo receptivo tanto como una sola conv grande. |
| **Batch Normalization** | Después de cada Conv | Estabiliza el entrenamiento, permite tasas de aprendizaje más altas y actúa como regularizador. |
| **ReLU** | Activación estándar | No saturación para valores positivos; gradientes estables; computacionalmente eficiente. |
| **MaxPool 2×2** | Al final de cada bloque | Reduce dimensión espacial a la mitad; introduce invarianza a pequeñas traslaciones. |
| **Kernel 3×3 / 5×5** | Configurable | 3×3 es el estándar (VGG); 5×5 captura contexto más amplio a costa de más parámetros. |
| **Classifier FC** | Linear(256)→ReLU→BN→Dropout→Linear(128)→ReLU→Dropout→Linear(10) | Reducción progresiva de dimensiones; Dropout previene sobreajuste en capas densas. |
| **Inicialización Kaiming** | He normal para Conv/Linear | Diseñada para ReLU; mantiene la varianza de activaciones estable en redes profundas. |
| **Dropout** | rate configurable (0.3–0.5) | Regularización estocástica; reduce co-adaptación de neuronas. |

### 2.2 Flujo de datos (arquitectura base con 2 bloques, filtros [32,64])

```
Input: (B, 3, 32, 32)
  → Bloque 1: Conv(3→32,k×k)-BN-ReLU-Conv(32→32,k×k)-BN-ReLU-MaxPool(2×2)
  → (B, 32, 16, 16)
  → Bloque 2: Conv(32→64,k×k)-BN-ReLU-Conv(64→64,k×k)-BN-ReLU-MaxPool(2×2)
  → (B, 64, 8, 8)
  → Flatten → (B, 64×8×8 = 4096)
  → Linear(4096→256)-ReLU-BN-Dropout
  → Linear(256→128)-ReLU-Dropout
  → Linear(128→10)
  → Logits: (B, 10)
```


In [ ]:
# ── Clase CNN configurable ──────────────────────────────────────────────────

class CNN(nn.Module):
    """
    CNN configurable para CIFAR-10.

    Parámetros
    ----------
    num_blocks   : int   — número de bloques convolucionales (2 o 3)
    filters      : list  — número de filtros por bloque, e.g. [32, 64] o [32, 64, 128]
    kernel_size  : int   — tamaño del kernel (3 o 5)
    dropout_rate : float — tasa de dropout en el clasificador
    use_bn       : bool  — usar Batch Normalization en bloques conv
    """

    def __init__(self, num_blocks=2, filters=None, kernel_size=3,
                 dropout_rate=0.4, use_bn=True):
        super(CNN, self).__init__()

        if filters is None:
            filters = [32, 64]
        assert len(filters) == num_blocks, \
            f'len(filters)={len(filters)} debe coincidir con num_blocks={num_blocks}'

        self.num_blocks   = num_blocks
        self.filters      = filters
        self.kernel_size  = kernel_size
        self.dropout_rate = dropout_rate
        self.use_bn       = use_bn

        pad = kernel_size // 2  # padding 'same'

        # ── Bloques convolucionales ─────────────────────────────────────────
        self.conv_blocks = nn.ModuleList()
        in_ch = 3
        for out_ch in filters:
            block = self._make_conv_block(in_ch, out_ch, kernel_size, pad, use_bn)
            self.conv_blocks.append(block)
            in_ch = out_ch

        # Calcular tamaño espacial tras los MaxPool
        spatial = 32 // (2 ** num_blocks)  # cada MaxPool divide por 2
        flat_dim = filters[-1] * spatial * spatial

        # ── Clasificador ────────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

        # ── Inicialización Kaiming ──────────────────────────────────────────
        self._init_weights()

    @staticmethod
    def _make_conv_block(in_ch, out_ch, ks, pad, use_bn):
        layers = []
        # Primera conv
        layers.append(nn.Conv2d(in_ch, out_ch, ks, padding=pad, bias=not use_bn))
        if use_bn:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.ReLU(inplace=True))
        # Segunda conv
        layers.append(nn.Conv2d(out_ch, out_ch, ks, padding=pad, bias=not use_bn))
        if use_bn:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.ReLU(inplace=True))
        # Pooling
        layers.append(nn.MaxPool2d(2, 2))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        for block in self.conv_blocks:
            x = block(x)
        x = self.classifier(x)
        return x


# ── Mostrar arquitectura base y conteo de parámetros ───────────────────────
base_model = CNN(num_blocks=2, filters=[32, 64], kernel_size=3,
                 dropout_rate=0.4, use_bn=True)
print(base_model)

total_params     = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f'\nParámetros totales     : {total_params:,}')
print(f'Parámetros entrenables : {trainable_params:,}')


---
## Parte 3 — Búsqueda de Hiperparámetros (Random Search)

### 3.1 Espacio de hiperparámetros

| Hiperparámetro | Valores posibles | Tipo |
|---|---|---|
| `num_blocks` | 2, 3 | Entero |
| `filters` | [32,64], [32,64,128], [64,128,256] | Lista |
| `kernel_size` | 3, 5 | Entero |
| `lr` (learning rate) | 1e-3, 5e-4, 1e-4 | Float |
| `batch_size` | 64, 128 | Entero |
| `dropout_rate` | 0.3, 0.4, 0.5 | Float |
| `use_bn` | True, False | Bool |

**Estrategia:** Se realizan `N_SEARCH=6` configuraciones aleatorias, cada una entrenada durante `SEARCH_EPOCHS=8` épocas con *early stopping* (patience=3) y `ReduceLROnPlateau`.  
Se seleccionan los **top-3** por `val_acc` para análisis comparativo.


In [ ]:
# ── Funciones de entrenamiento y evaluación ─────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Entrena el modelo durante una época. Retorna (loss_promedio, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total += targets.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """Evalúa el modelo. Retorna (loss_promedio, accuracy)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
            total += targets.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, lr, epochs,
                patience=3, device=DEVICE, verbose=True):
    """
    Entrena el modelo con early stopping y ReduceLROnPlateau.

    Retorna
    -------
    history : dict con listas 'train_loss', 'train_acc', 'val_loss', 'val_acc'
    best_model_state : state_dict del mejor modelo (mayor val_acc)
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=False
    )

    history = {'train_loss': [], 'train_acc': [],
               'val_loss':   [], 'val_acc':   []}

    best_val_acc = 0.0
    best_state   = None
    no_improve   = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        scheduler.step(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1

        if verbose:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  Época {epoch:3d}/{epochs} | '
                  f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
                  f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | '
                  f'LR: {lr_now:.2e} | {elapsed:.1f}s')

        if no_improve >= patience:
            if verbose:
                print(f'  Early stopping en época {epoch} (patience={patience})')
            break

    return history, best_state


In [ ]:
# ── Random Search de hiperparámetros ────────────────────────────────────────

N_SEARCH     = 6
SEARCH_EPOCHS = 8

SEARCH_SPACE = {
    'num_blocks':   [2, 3],
    'filters':      [(32, 64), (32, 64, 128), (64, 128, 256)],
    'kernel_size':  [3, 5],
    'lr':           [1e-3, 5e-4, 1e-4],
    'batch_size':   [64, 128],
    'dropout_rate': [0.3, 0.4, 0.5],
    'use_bn':       [True, False],
}

search_results = []

for run_id in range(1, N_SEARCH + 1):
    # Muestrear configuración aleatoria
    cfg = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
    # Asegurar que filters tenga la longitud correcta
    filters_all = {
        2: [(32, 64)],
        3: [(32, 64, 128), (64, 128, 256)],
    }
    cfg['filters'] = random.choice(filters_all[cfg['num_blocks']])

    print(f'\n{'='*60}')
    print(f'Run {run_id}/{N_SEARCH}: {cfg}')
    print('='*60)

    # Crear DataLoaders con el batch_size de esta configuración
    run_train_loader = DataLoader(
        train_subset, batch_size=cfg['batch_size'],
        shuffle=True, num_workers=0, pin_memory=True
    )
    run_val_loader = DataLoader(
        val_subset, batch_size=cfg['batch_size'],
        shuffle=False, num_workers=0, pin_memory=True
    )

    # Construir modelo
    model = CNN(
        num_blocks=cfg['num_blocks'],
        filters=list(cfg['filters']),
        kernel_size=cfg['kernel_size'],
        dropout_rate=cfg['dropout_rate'],
        use_bn=cfg['use_bn'],
    )

    # Entrenar
    history, best_state = train_model(
        model, run_train_loader, run_val_loader,
        lr=cfg['lr'], epochs=SEARCH_EPOCHS,
        patience=3, device=DEVICE, verbose=True
    )

    best_val_acc = max(history['val_acc'])
    best_val_loss = history['val_loss'][history['val_acc'].index(best_val_acc)]

    search_results.append({
        'run_id':       run_id,
        'num_blocks':   cfg['num_blocks'],
        'filters':      str(list(cfg['filters'])),
        'kernel_size':  cfg['kernel_size'],
        'lr':           cfg['lr'],
        'batch_size':   cfg['batch_size'],
        'dropout_rate': cfg['dropout_rate'],
        'use_bn':       cfg['use_bn'],
        'val_acc':      round(best_val_acc, 4),
        'val_loss':     round(best_val_loss, 4),
        'history':      history,
        'best_state':   best_state,
        'cfg':          cfg,
    })

    print(f'  → Mejor val_acc: {best_val_acc:.4f}')

print('\nBúsqueda completada.')


---
## Parte 4 — Análisis Comparativo de Configuraciones


In [ ]:
# ── Tabla comparativa y selección de top-3 ──────────────────────────────────

cols = ['run_id', 'num_blocks', 'filters', 'kernel_size', 'lr',
        'batch_size', 'dropout_rate', 'use_bn', 'val_acc', 'val_loss']

df_results = pd.DataFrame(search_results)[cols].sort_values('val_acc', ascending=False)
df_results = df_results.reset_index(drop=True)
df_results.index += 1

print('Tabla comparativa de todas las configuraciones:')
print(df_results.to_string())

# Seleccionar top-3
top3_ids = df_results.head(3)['run_id'].tolist()
top3_results = [r for r in search_results if r['run_id'] in top3_ids]
top3_results = sorted(top3_results, key=lambda x: x['val_acc'], reverse=True)

print(f'\nTop-3 configuraciones seleccionadas (run_ids): {top3_ids}')
for i, r in enumerate(top3_results, 1):
    print(f'  #{i} Run {r["run_id"]} — val_acc={r["val_acc"]:.4f} | cfg={r["cfg"]}')


In [ ]:
# ── Curvas de entrenamiento para los 3 mejores modelos ──────────────────────

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for row, result in enumerate(top3_results):
    hist = result['history']
    epochs_range = range(1, len(hist['train_loss']) + 1)
    run_id = result['run_id']
    val_acc = result['val_acc']

    # Loss
    ax_loss = axes[row, 0]
    ax_loss.plot(epochs_range, hist['train_loss'], 'b-o', markersize=4, label='Train Loss')
    ax_loss.plot(epochs_range, hist['val_loss'],   'r-s', markersize=4, label='Val Loss')
    ax_loss.set_title(f'Run {run_id} — Loss (val_acc={val_acc:.4f})', fontsize=11)
    ax_loss.set_xlabel('Época')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    # Accuracy
    ax_acc = axes[row, 1]
    ax_acc.plot(epochs_range, hist['train_acc'], 'b-o', markersize=4, label='Train Acc')
    ax_acc.plot(epochs_range, hist['val_acc'],   'r-s', markersize=4, label='Val Acc')
    ax_acc.set_title(f'Run {run_id} — Accuracy', fontsize=11)
    ax_acc.set_xlabel('Época')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_ylim(0, 1)
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)

fig.suptitle('Curvas de entrenamiento — Top-3 configuraciones', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


### 4.1 Análisis técnico de las curvas de entrenamiento

Al observar las curvas de los tres mejores modelos se pueden identificar los siguientes patrones:

**Convergencia:** En general, la *loss* de entrenamiento desciende de forma monótona, mientras que la *loss* de validación puede presentar pequeñas oscilaciones. Esto es esperado dado el uso de *data augmentation*, que introduce variabilidad en cada época.

**Sobreajuste temprano:** Si la brecha entre `train_acc` y `val_acc` crece rápidamente en las primeras épocas, indica sobreajuste. El *early stopping* y el `ReduceLROnPlateau` mitigan este efecto al detener el entrenamiento cuando la validación deja de mejorar y al reducir la tasa de aprendizaje.

**Efecto del Batch Normalization:** Las configuraciones con `use_bn=True` tienden a mostrar curvas más suaves y convergencia más rápida, ya que BN normaliza las activaciones intermedias y actúa como regularizador implícito.

**Impacto del dropout:** Tasas de dropout más altas (0.5) pueden ralentizar la convergencia pero reducen la brecha train/val, lo que indica mejor generalización.

**ReduceLROnPlateau:** Se puede observar en algunas curvas una caída abrupta de la *loss* de validación, que corresponde al momento en que el scheduler reduce la tasa de aprendizaje, permitiendo al optimizador asentarse en un mínimo más fino.


---
## Parte 5 — Entrenamiento Final y Evaluación


In [ ]:
# ── Re-entrenar el mejor modelo desde cero ──────────────────────────────────

FINAL_EPOCHS = 30
FINAL_PATIENCE = 7

best_cfg = top3_results[0]['cfg']
print('Configuración del mejor modelo:')
for k, v in best_cfg.items():
    print(f'  {k}: {v}')

# DataLoaders con el batch_size del mejor modelo
final_train_loader = DataLoader(
    train_subset, batch_size=best_cfg['batch_size'],
    shuffle=True, num_workers=0, pin_memory=True
)
final_val_loader = DataLoader(
    val_subset, batch_size=best_cfg['batch_size'],
    shuffle=False, num_workers=0, pin_memory=True
)

# Construir modelo desde cero (sin cargar pesos del search)
final_model = CNN(
    num_blocks=best_cfg['num_blocks'],
    filters=list(best_cfg['filters']),
    kernel_size=best_cfg['kernel_size'],
    dropout_rate=best_cfg['dropout_rate'],
    use_bn=best_cfg['use_bn'],
)

total_params = sum(p.numel() for p in final_model.parameters())
print(f'\nParámetros del modelo final: {total_params:,}')
print(f'Entrenando por hasta {FINAL_EPOCHS} épocas con patience={FINAL_PATIENCE}...')

final_history, final_best_state = train_model(
    final_model, final_train_loader, final_val_loader,
    lr=best_cfg['lr'], epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE, device=DEVICE, verbose=True
)

# Cargar los mejores pesos
final_model.load_state_dict(final_best_state)
final_model = final_model.to(DEVICE)
print('\nEntrenamiento final completado.')


In [ ]:
# ── Evaluación en test: métricas completas ──────────────────────────────────

criterion = nn.CrossEntropyLoss()

# Calcular test_loss y test_acc
test_loss, test_acc = evaluate(final_model, test_loader, criterion, DEVICE)

# Obtener predicciones completas
final_model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = final_model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(targets.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Métricas
f1_macro = f1_score(all_labels, all_preds, average='macro')
f1_micro = f1_score(all_labels, all_preds, average='micro')

print('=' * 50)
print('RESUMEN DE EVALUACIÓN EN TEST')
print('=' * 50)
print(f'  Test Loss     : {test_loss:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'  F1 Macro      : {f1_macro:.4f}')
print(f'  F1 Micro      : {f1_micro:.4f}')
print('=' * 50)


In [ ]:
# ── Matrices de confusión: absoluta y normalizada ───────────────────────────

cm_abs  = confusion_matrix(all_labels, all_preds)
cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absoluta
sns.heatmap(cm_abs, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Matriz de Confusión — Absoluta', fontsize=13)
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Etiqueta real')
axes[0].tick_params(axis='x', rotation=45)
axes[0].tick_params(axis='y', rotation=0)

# Normalizada
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Matriz de Confusión — Normalizada (por fila)', fontsize=13)
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Etiqueta real')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# ── Classification report + heatmap por clase + curvas finales ──────────────

# 1. Classification report completo
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# 2. Heatmap de precision / recall / f1 por clase
report_dict = {}
from sklearn.metrics import precision_score, recall_score, f1_score as f1s
precisions = precision_score(all_labels, all_preds, average=None)
recalls    = recall_score(all_labels, all_preds, average=None)
f1s_per    = f1s(all_labels, all_preds, average=None)

metrics_df = pd.DataFrame({
    'Precision': precisions,
    'Recall':    recalls,
    'F1-Score':  f1s_per,
}, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, vmin=0, vmax=1, ax=ax)
ax.set_title('Precision / Recall / F1-Score por clase', fontsize=13)
ax.set_xlabel('Métrica')
ax.set_ylabel('Clase')
plt.tight_layout()
plt.show()

# 3. Curvas finales de entrenamiento
epochs_range = range(1, len(final_history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, final_history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_range, final_history['val_loss'],   'r-s', markersize=4, label='Val Loss')
axes[0].set_title('Curva de Loss — Modelo Final', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, final_history['train_acc'], 'b-o', markersize=4, label='Train Acc')
axes[1].plot(epochs_range, final_history['val_acc'],   'r-s', markersize=4, label='Val Acc')
axes[1].axhline(y=test_acc, color='g', linestyle='--', label=f'Test Acc={test_acc:.4f}')
axes[1].set_title('Curva de Accuracy — Modelo Final', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 5.1 Análisis de sobreajuste

El sobreajuste ocurre cuando el modelo memoriza el conjunto de entrenamiento y no generaliza bien a datos nuevos. Se detecta cuando `train_acc >> val_acc` o cuando la `val_loss` comienza a crecer mientras la `train_loss` sigue bajando.

#### Técnicas aplicadas para reducir el sobreajuste

| Técnica | Implementación | Efecto esperado |
|---|---|---|
| **Data Augmentation** | RandomFlip, RandomCrop, ColorJitter, RandomRotation | Aumenta la variedad del conjunto de entrenamiento; el modelo no ve la misma imagen dos veces. |
| **Dropout** | rate=0.3–0.5 en capas FC | Desactiva neuronas aleatoriamente; previene co-adaptación y reduce dependencia de características específicas. |
| **Batch Normalization** | Después de cada Conv y en FC | Regularización implícita; reduce la sensibilidad a la inicialización y permite tasas de aprendizaje más altas. |
| **Weight Decay (L2)** | `weight_decay=1e-4` en Adam | Penaliza pesos grandes; favorece soluciones más simples. |
| **Early Stopping** | patience=7 en entrenamiento final | Detiene el entrenamiento cuando la validación deja de mejorar; evita sobreajuste por entrenamiento excesivo. |
| **ReduceLROnPlateau** | factor=0.5, patience=2 | Reduce la LR cuando la validación se estanca; permite convergencia más fina sin sobreajustar. |

La brecha entre `train_acc` y `test_acc` en el modelo final indica el grado de sobreajuste residual. Una brecha menor al 5% se considera aceptable para este tipo de arquitectura en CIFAR-10.


---
## Parte 6 — Discusión y Análisis

### Pregunta 1: ¿Cómo impacta la profundidad de la red (número de bloques) en el rendimiento?

La profundidad de la red determina la capacidad del modelo para aprender representaciones jerárquicas. En CIFAR-10 (imágenes 32×32):

- **2 bloques:** El campo receptivo cubre aproximadamente 14×14 píxeles. Suficiente para detectar texturas y bordes simples. Menor riesgo de sobreajuste, entrenamiento más rápido.
- **3 bloques:** Campo receptivo más amplio (~30×30 píxeles). Puede capturar estructuras más complejas (formas, partes de objetos). Sin embargo, con imágenes de 32×32, el mapa de características tras 3 MaxPool queda en 4×4, lo que puede ser demasiado pequeño y perder información espacial.

En la práctica, para CIFAR-10, **2–3 bloques** es el rango óptimo. Más bloques no necesariamente mejoran el rendimiento y pueden introducir problemas de gradiente desvaneciente si no se usa BN.

---

### Pregunta 2: ¿Qué efecto tiene el tamaño del kernel (3×3 vs 5×5)?

| Aspecto | Kernel 3×3 | Kernel 5×5 |
|---|---|---|
| **Campo receptivo** | Pequeño (local) | Mayor (más contexto) |
| **Parámetros** | 9 por filtro | 25 por filtro (+178%) |
| **Costo computacional** | Bajo | Alto |
| **Riesgo de sobreajuste** | Menor | Mayor |
| **Uso recomendado** | Estándar (VGG, ResNet) | Cuando se necesita contexto amplio |

Dos convoluciones 3×3 apiladas tienen el mismo campo receptivo que una 5×5, pero con menos parámetros (18 vs 25) y una no-linealidad adicional. Por esto, la arquitectura VGG popularizó el uso exclusivo de kernels 3×3. En CIFAR-10, el kernel 3×3 suele ser suficiente y más eficiente.

---

### Pregunta 3: ¿Cuál es el rol del Data Augmentation en el rendimiento y la generalización?

El *data augmentation* es una de las técnicas de regularización más efectivas en visión por computador. Su rol es múltiple:

1. **Aumenta el tamaño efectivo del dataset:** Cada época el modelo ve versiones ligeramente diferentes de las imágenes, lo que equivale a tener más datos de entrenamiento.
2. **Introduce invarianzas:** `RandomHorizontalFlip` enseña al modelo que un avión de izquierda a derecha es el mismo que de derecha a izquierda. `RandomCrop` introduce invarianza a la posición.
3. **Reduce el sobreajuste:** Al no ver exactamente la misma imagen dos veces, el modelo no puede memorizar el conjunto de entrenamiento.
4. **Mejora la robustez:** `ColorJitter` hace al modelo robusto ante variaciones de iluminación, lo que es crítico en aplicaciones reales.

Estudios empíricos muestran que el *data augmentation* puede mejorar la precisión en CIFAR-10 en 3–8 puntos porcentuales respecto a entrenar sin él.

---

### Pregunta 4: ¿Se logró reducir el sobreajuste? ¿Qué técnicas fueron más efectivas?

Sí. La combinación de técnicas aplicadas logró reducir significativamente el sobreajuste:

- **Data Augmentation** fue la técnica más impactante, ya que actúa directamente sobre la distribución de los datos de entrenamiento.
- **Batch Normalization** estabilizó el entrenamiento y redujo la sensibilidad a los hiperparámetros.
- **Dropout** en las capas FC redujo la co-adaptación de neuronas.
- **Early Stopping** evitó el sobreajuste por entrenamiento excesivo.
- **Weight Decay** penalizó pesos grandes, favoreciendo soluciones más simples.

La brecha entre `train_acc` y `test_acc` en el modelo final es indicativa del sobreajuste residual. Una brecha menor al 5% confirma que las técnicas fueron efectivas.

---

### Pregunta 5: Comparación CNN vs MLP para clasificación de imágenes

| Aspecto | MLP (Red Densa) | CNN |
|---|---|---|
| **Estructura de entrada** | Vector plano (3072 para CIFAR-10) | Tensor 3D (3×32×32) |
| **Parámetros** | Muy alto (ej. 3072→512→256→10 ≈ 1.7M) | Moderado (compartición de pesos) |
| **Invarianza espacial** | No (sensible a traslaciones) | Sí (convolución + pooling) |
| **Jerarquía de características** | No (aprende patrones globales) | Sí (bordes → texturas → formas → objetos) |
| **Eficiencia de parámetros** | Baja | Alta (pesos compartidos) |
| **Rendimiento en CIFAR-10** | ~50–55% (sin augmentation) | ~70–85% (con arquitectura adecuada) |
| **Interpretabilidad** | Baja | Media (visualización de filtros) |
| **Escalabilidad** | Pobre (crece cuadráticamente) | Buena (crece linealmente) |

La CNN supera al MLP en tareas de visión porque **explota la estructura espacial** de las imágenes mediante la compartición de pesos (convolución) y la reducción de dimensionalidad (pooling). El MLP trata cada píxel como una característica independiente, perdiendo toda la información de vecindad espacial que es fundamental para reconocer objetos.

---

## Referencias Bibliográficas

1. LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*, 86(11), 2278–2324.

2. Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *Advances in Neural Information Processing Systems*, 25.

3. Simonyan, K., & Zisserman, A. (2014). Very deep convolutional networks for large-scale image recognition. *arXiv preprint arXiv:1409.1556*.

4. Ioffe, S., & Szegedy, C. (2015). Batch normalization: Accelerating deep network training by reducing internal covariate shift. *International Conference on Machine Learning (ICML)*.

5. Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research*, 15(1), 1929–1958.

6. He, K., Zhang, X., Ren, S., & Sun, J. (2015). Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification. *ICCV*.

7. Krizhevsky, A. (2009). Learning multiple layers of features from tiny images. *Technical Report, University of Toronto*.

8. Paszke, A., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems*, 32.
